# Piloto de fine-tuning — filtrar y seleccionar el español sintético

Se ejecuta **después** del cuaderno 01, para una voz cada vez (`VOZ`). Mide cada clip sintético y se queda
con los buenos:

1. **WER** con Whisper large-v3: descarta los que no dicen exactamente el texto (alucinaciones, saltos).
2. **Audio que sobra al final**: descarta los que tienen más de 0,8 s después de la última palabra. Ahí
   suelen estar los murmullos o sílabas alucinadas que el WER no ve. (El del principio no se puede medir:
   Whisper marca siempre la primera palabra en 0,0 s.)
3. **Velocidad de habla** en caracteres por segundo: descarta los cortados o estirados.
4. **Similitud** con las referencias de la voz (ECAPA): no descarta, ordena. Entre dos clips buenos de la
   misma frase gana el que más se parece.

Después elige **una versión por frase** hasta los minutos objetivo: por defecto, los que hacen falta para
que el español sea el **40 %** frente al inglés real de su dataset (`datasets/<voz>/`).

Salida en `fine_tunned_naval/piloto/sintetico/<voz>/` (la de Naval, en `piloto/sintetico/`):
`medidas.jsonl` (todas las medidas), `seleccion.jsonl` (los elegidos, con texto, duración y medidas) e
`informe.md`. Solo se usa el profesor Chatterbox V2: XTTS quedó descartado.

**GPU:** T4. **Tiempo estimado:** unos 15 min por voz. Uso: celda 1, reiniciar sesión, y el resto en orden.
Lo ya medido no se vuelve a medir. Para otra voz, cambia `VOZ` y ejecuta de la celda 2 en adelante.

In [ ]:
# 1. Instalación (después: Entorno de ejecución → Reiniciar sesión)
!pip install -q jiwer speechbrain librosa soundfile num2words

In [ ]:
# 2. Preparar
VOZ = "nivi"          # nivi, brett o deutsch (naval ya está hecho)

import os, sys, subprocess, json
from pathlib import Path
RAMA = "claude/quirky-dirac-x18c3n"
if os.path.isdir("/content/repo"):
    subprocess.run(["git", "-C", "/content/repo", "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", RAMA, "https://github.com/DreiSup/spanishnav.al-podcast", "/content/repo"], check=True)
sys.path[:0] = ["/content/repo/criba", "/content/repo/piloto"]
import comun, medir, rutas

comun.montar_drive()
comun.FRASES = Path("/content/repo/piloto/frases_sinteticas.json")
SINT = rutas.sintetico(comun.DRIVE, VOZ)
ITEMS, _ = comun.cargar_items()
TEXTO = {it["id"]: it["frases"][0] for it in ITEMS}
PROFESORES = ["chatterbox_v2"]
CLIPS = [(v, i) for v in PROFESORES for i in TEXTO if (SINT / v / f"{i}.wav").exists()]
print(VOZ, "| clips a medir:", len(CLIPS))
if not CLIPS:
    raise SystemExit(f"No hay clips en {SINT / 'chatterbox_v2'}: ejecuta antes el cuaderno 01 con esta voz.")

In [ ]:
# 3. Medir: WER, velocidad de habla y similitud con la voz
MEDIDAS = SINT / "medidas.jsonl"
hecho = medir.cargar(MEDIDAS)
pendientes = [(v, i) for v, i in CLIPS if (v, i) not in hecho]
print("Pendientes:", len(pendientes))
if pendientes:
    asr, ecapa = medir.Asr(), medir.Ecapa()
    ref_voz = ecapa.referencia(sorted((comun.REFS / VOZ).glob(f"{VOZ}_ref0*.wav")))
    for n, (v, i) in enumerate(pendientes, 1):
        wav = SINT / v / f"{i}.wav"
        hyp, inicio, fin = asr.transcribe(wav)
        dur = medir.duracion(wav)
        fila = {"variante": v, "id": i, "wer": medir.wer(TEXTO[i], hyp), "transcripcion": hyp,
                "segundos": round(dur, 2), "cps": round(len(TEXTO[i]) / dur, 1),
                "sobra_inicio": round(inicio, 2) if inicio is not None else None,
                "sobra_final": round(dur - fin, 2) if fin is not None else None,
                "similitud": ecapa.similitud(wav, ref_voz)}
        medir.anadir(MEDIDAS, fila)
        hecho[(v, i)] = fila
        if n % 50 == 0:
            print(f"  {n}/{len(pendientes)}")
    del asr, ecapa
    comun.liberar()
print("Medidos:", len(hecho))

In [ ]:
# 4. Filtrar y seleccionar
WER_MAX = 0.05        # tolera una palabra mal transcrita por el ASR en una frase larga
CPS = (9, 22)         # fuera de ahí, el clip está cortado o estirado
SOBRA_MAX = 0.8       # segundos de audio después de la última palabra: más suele ser un murmullo alucinado
FRAC_ES = 0.4         # el español, 40 % de los minutos del entrenamiento
en_train = [json.loads(l) for l in (rutas.dataset(comun.DRIVE, VOZ) / "metadata.jsonl").read_text(
    encoding="utf-8").splitlines() if l.strip()]
MIN_INGLES = sum(f["dur"] for f in en_train if f["split"] == "train") / 60
MIN_OBJETIVO = MIN_INGLES * FRAC_ES / (1 - FRAC_ES)

def limpio(f):
    return f["sobra_final"] is not None and f["sobra_final"] <= SOBRA_MAX

filas = list(hecho.values())
buenos = [f for f in filas if f["wer"] <= WER_MAX and CPS[0] <= f["cps"] <= CPS[1] and limpio(f)]
# Una versión por frase: la de menor WER y, a igualdad, la que más se parece a la voz
mejor = {}
for f in sorted(buenos, key=lambda f: (f["wer"], -f["similitud"])):
    mejor.setdefault(f["id"], f)
ordenados = sorted(mejor.values(), key=lambda f: (f["wer"], -f["similitud"]))
relativa = SINT.relative_to(comun.DRIVE)
seleccion, minutos = [], 0.0
for f in ordenados:
    if minutos >= MIN_OBJETIVO:
        break
    seleccion.append({**f, "texto": TEXTO[f["id"]], "audio": f"{relativa}/{f['variante']}/{f['id']}.wav"})
    minutos += f["segundos"] / 60

(SINT / "seleccion.jsonl").write_text("".join(json.dumps(s, ensure_ascii=False) + "\n" for s in seleccion),
                                      encoding="utf-8")
lineas = [f"# Español sintético de {VOZ}: filtro y selección", "",
          f"Umbrales: WER <= {WER_MAX}, caracteres/s entre {CPS[0]} y {CPS[1]}, como mucho {SOBRA_MAX} s de audio "
          f"después de la última palabra. Objetivo: {MIN_OBJETIVO:.1f} min, el {FRAC_ES:.0%} frente a "
          f"{MIN_INGLES:.1f} min de inglés real.", "",
          "| Profesor | Clips | Pasan el filtro | Elegidos | Min elegidos | WER medio | Similitud media |",
          "|---|---|---|---|---|---|---|"]
for v in PROFESORES:
    tot = [f for f in filas if f["variante"] == v]
    ok = [f for f in buenos if f["variante"] == v]
    sel = [f for f in seleccion if f["variante"] == v]
    lineas.append(f"| {v} | {len(tot)} | {len(ok)} | {len(sel)} | {sum(f['segundos'] for f in sel) / 60:.1f} | "
                  f"{sum(f['wer'] for f in tot) / max(len(tot), 1):.3f} | "
                  f"{sum(f['similitud'] for f in tot) / max(len(tot), 1):.3f} |")
lineas += ["", f"**Total elegido: {len(seleccion)} clips, {minutos:.1f} min**, de {len(mejor)} frases con al menos "
           f"una versión buena."]
if minutos < MIN_OBJETIVO:
    lineas += ["", f"Aviso: faltan {MIN_OBJETIVO - minutos:.1f} min para el objetivo. El LoRA mezclará con lo que "
               f"hay y el español pesará algo menos del {FRAC_ES:.0%}."]
informe = "\n".join(lineas)
(SINT / "informe.md").write_text(informe + "\n", encoding="utf-8")
print(informe)

In [ ]:
# 5. Subir a Drive y desconectar
comun.desconectar()